In [42]:
import pandas as pd

In [43]:
customers = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/customers.csv')
# print(customers.head())
# print(customers.info())
# print(customers.isnull().sum())

In [44]:
accounts = pd.read_excel('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/accounts_table.xlsx')
# print(accounts.head())
# print(accounts.info())
# print(accounts.isnull().sum())

In [45]:
branches = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/branches.csv')
# print(branches.head())
# print(branches.info())
# print(branches.isnull().sum())

In [46]:
employees = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/employees.csv')
# print(employees.head())
# print(employees.info())
# print(employees.isnull().sum())

In [47]:
loan_payments = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/loan_payments.csv')
# print(loan_payments.head())
# print(loan_payments.info())
# print(loan_payments.isnull().sum())

In [48]:
loans = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/loans.csv')
# print(loans.head())
# print(loans.info())
# print(loans.isnull().sum())

In [49]:
transactions = pd.read_csv('D:/ADITYA/Projects/FinSight_Analytics/datasets_from_sql/transactions.csv')
# print(transactions.head())
# print(transactions.info())
# print(transactions.isnull().sum())

In [50]:
account_summary = accounts.groupby("customer_id").agg(
    account_count=("account_id", "count"),
    total_balance=("balance", "sum")
).reset_index()

transaction_data = transactions.merge(accounts[["account_id", "customer_id"]], on="account_id")

transaction_summary = transaction_data.groupby("customer_id").agg(
    total_debit=("amount", lambda x: x[transaction_data.loc[x.index, "transaction_type"] == "debit"].sum()),
    total_credit=("amount", lambda x: x[transaction_data.loc[x.index, "transaction_type"] == "credit"].sum()),
    transaction_count=("transaction_id", "count")
).reset_index()

loan_summary = loans.groupby("customer_id").agg(
    total_loan_amount=("loan_amount", "sum"),
    loan_count=("loan_id", "count")
).reset_index()

missed_summary = loans[["loan_id", "customer_id"]].merge(loan_payments, on="loan_id")

missed_summary = missed_summary.groupby("customer_id").agg(
    missed_payments=("payment_status", lambda x: (x == "missed").sum())
).reset_index()

In [51]:
customer_df = customers.merge(account_summary, on="customer_id", how="left")
customer_df = customer_df.merge(transaction_summary, on="customer_id", how="left")
customer_df = customer_df.merge(loan_summary, on="customer_id", how="left")
customer_df = customer_df.merge(missed_summary, on="customer_id", how="left")

customer_df = customer_df.fillna(0)

In [52]:
def customer_segment(balance):
    if balance >= 500000:
        return "High Value"
    elif balance >= 100000:
        return "Medium Value"
    else:
        return "Low Value"

customer_df["customer_value_segment"] = customer_df["total_balance"].apply(customer_segment)


def risk_segment(row):
    if row["missed_payments"] >= 3:
        return "High Risk"
    elif row["missed_payments"] >= 1 or row["total_loan_amount"] > row["income"] * 3:
        return "Medium Risk"
    else:
        return "Low Risk"

customer_df["risk_segment"] = customer_df.apply(risk_segment, axis=1)

In [53]:
customer_df.to_csv(OUTPUT_DIR / "customer_analytics.csv", index=False)

In [54]:
# print(customer_df.head())
# print(customer_df.info())